<h2 style="color:#2563eb;">Hafta 4 - Görev 1</h2>

<p>
<strong>Part 2 ile başla.</strong> Önceki üç harfi bağlam alan veri setini kur
(<strong>X: 3 harf indeksi</strong>, <strong>Y: sıradaki harf</strong>).
</p>

<p>
<strong>Embedding tablosunu (27x2)</strong> oluştur, indeksleme ile embedding'leri çek.
</p>

<p style="color:#6b7280;">
<em>(Part 2, 9:03 - 18:35)</em>
</p>

<hr>



### Bağlam Penceresi (`block_size = 3`) ile $X$ ve $Y$ Oluşturma

In [2]:
import torch
import torch.nn.functional as F

words = open("../hafta3/names.txt", "r").read().splitlines()
chars = sorted(list(set(''.join(words))))
s2i = {s: i + 1 for i, s in enumerate(chars)}
s2i['.'] = 0
i2s = {i: s for s, i in s2i.items()}


block_size = 3


def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = s2i[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

X, Y = build_dataset(words)

print(f'X boyutu: {X.shape}')
print(f'Y boyutu: {Y.shape}')


X boyutu: torch.Size([228146, 3])
Y boyutu: torch.Size([228146])


### Embedding Tablosu ($C$) ve İndeksleme Mantığı

In [3]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)


emb = C[X]
print(f'emb (C[X]) boyutu: {emb.shape}')

emb (C[X]) boyutu: torch.Size([228146, 3, 2])


<h2 style="color:#2563eb;">Görev 2</h2>

<p>
<strong>Gizli katmanı ve çıkış katmanını kur:</strong>
embedding'leri düzleştir, W1 ve b1 ile tanh, W2 ve b2 ile logits.
</p>

<p>
Loss'u geçen haftaki gibi elle hesapla, sonra
<strong style="color:#16a34a;">F.cross_entropy</strong> ile aynı sonucu aldığını göster
ve neden onu tercih ettiğimizi videodan anla.
</p>

### Model Parametrelerini Tanımlama

In [4]:
import torch
import torch.nn.functional as F


g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)


W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)


W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]


for p in parameters:
  p.requires_grad = True

print(f"Toplam parametre sayısı: {sum(p.nelement() for p in parameters)}")

Toplam parametre sayısı: 3481


### İleri Yayılım ve Kayıp Karşılaştırması

In [5]:
emb = C[X]

h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2

counts = logits.exp()

probs = counts / counts.sum(1, keepdims=True)

loss_manual = -probs[torch.arange(X.shape[0]), Y].log().mean()

loss_builtin = F.cross_entropy(logits, Y)

print(f'Elle hesaplanan loss : {loss_manual.item(): .4f}')
print(f'F.cross_entropy ile hesaplanan loss : {loss_builtin.item(): .4f}')




Elle hesaplanan loss :  19.5052
F.cross_entropy ile hesaplanan loss :  19.5052
